# ProverbGap: Multi-Language Shortcut Audit (Pilot v6.0 — Rate Limit Fixes)
**Fixes in v6.0:**
1. **Rate Limit 429 Handling:** Added exponential backoff and automatic key rotation retries to prevent `API Error 429` (Too Many Requests).
2. **Throttling:** Added a small delay between requests to keep Groq and Cerebras APIs happy.
3. **Complete Transparency:** Output CSVs contain `correct_text`, reasoning logs, and actual generated distractors (not fallback "Alt A" placeholders).

In [ ]:
# Cell 1: Setup and API Key Loading
import subprocess, sys, time, os, json, re, random
from pathlib import Path
!pip install -q pandas requests
import pandas as pd
import requests

def _load_secrets():
    keys = {}
    try:
        from kaggle_secrets import UserSecretsClient
        s = UserSecretsClient()
        def get(name):
            try: 
                val = s.get_secret(name)
                return val if val and val.strip() else None
            except: return None
        keys['GROQ'] = [get(f'GROQ_API_KEY{"_"+str(i) if i>1 else ""}') for i in range(1,9)]
        keys['GROQ'] = [k for k in keys['GROQ'] if k]
        keys['CEREBRAS'] = [get(f'CEREBRAS_API_KEY{"_"+str(i) if i>1 else ""}') for i in range(1,3)]
        keys['CEREBRAS'] = [k for k in keys['CEREBRAS'] if k]
    except ImportError:
        keys['GROQ'] = [os.environ.get('GROQ_API_KEY')] if os.environ.get('GROQ_API_KEY') else []
        keys['CEREBRAS'] = [os.environ.get('CEREBRAS_API_KEY')] if os.environ.get('CEREBRAS_API_KEY') else []
    return keys

KEYS = _load_secrets()
OUT_DIR = Path('/kaggle/working')
OUT_DIR.mkdir(exist_ok=True)
print(f"Setup complete. Loaded {len(KEYS.get('GROQ',[]))} Groq keys and {len(KEYS.get('CEREBRAS',[]))} Cerebras keys.")

In [ ]:
# Cell 2: Adversarial Prompts & Parser
import re, json
ADVERSARIAL_CONSTRAINTS = (
    '\nCRITICAL ADVERSARIAL REQUIREMENTS:\n'
    '1. Word Count: Each distractor MUST be within +/- 10% of the word count of the correct answer.\n'
    '2. Complexity: Match the grammatical complexity.\n'
    '3. Tone: Match the formality and metaphor presence perfectly.\n'
    '4. Lexical Overlap: Distractors MUST use keywords from the proverb.'
)

def get_sys_prompt(task, strategy, lang):
    bt = "cultural interpretation" if task == 'b' else "literal translation"
    base = f'Expert in {bt}. Generate 3 distractors indistinguishable to a blind evaluator. {ADVERSARIAL_CONSTRAINTS}'
    if strategy == 'zs': return base + '\nReturn ONLY JSON array of 3 strings.'
    elif strategy == 'cot_en': return base + f'\nReason in ENGLISH first. Output JSON: {{"reasoning": "...", "distractors": ["d1", "d2", "d3"]}}'
    else: 
        rl = "English" if lang == "English" else lang
        return base + f'\nReason in {rl} first. Output JSON: {{"reasoning": "...", "distractors": ["d1", "d2", "d3"]}}'

def parse_response(raw):
    if not raw: return "No response", None
    try:
        clean = re.sub(r'```json|```', '', raw).strip()
        d = json.loads(clean)
        reasoning = d.get('reasoning', "No reasoning provided")
        dists = d.get('distractors', d if isinstance(d, list) else None)
        if dists and len(dists) >= 3: return reasoning, [str(x).strip() for x in dists[:3]]
    except: pass
    m = re.search(r'\[.*?\]', raw, re.DOTALL)
    if m:
        try: 
            res = json.loads(m.group())
            if len(res) >= 3: return "Regex Parse", [str(x).strip() for x in res[:3]]
        except: pass
    return "Parse Failed", None

def assemble_mcq(correct, distractors, item_id):
    if not distractors or len(distractors) < 3: distractors = ["API Failure A", "API Failure B", "API Failure C"]
    choices = list(distractors[:3]) + [correct]
    random.Random(42 + item_id).shuffle(choices)
    labels = ['A', 'B', 'C', 'D']
    ans = labels[choices.index(correct)]
    return {f'Choice_{l}': choices[i] for i, l in enumerate(labels)}, ans

In [ ]:
# Cell 3: Robust API Callers with Auto-Retry
_indices = {k: 0 for k in KEYS}
def call_api(provider, sys, user, model, max_retries=12):
    if not KEYS.get(provider): return None
    
    for attempt in range(max_retries):
        key = KEYS[provider][_indices[provider] % len(KEYS[provider])]
        _indices[provider] += 1
        url = 'https://api.groq.com/openai/v1/chat/completions' if provider=='GROQ' else 'https://api.cerebras.ai/v1/chat/completions'
        try:
            r = requests.post(url, headers={'Authorization': f'Bearer {key}'}, json={
                'model': model, 'messages': [{'role':'system','content':sys}, {'role':'user','content':user}],
                'temperature': 0.7 if 'guess' not in sys.lower() else 0.0, 'max_tokens': 1200
            }, timeout=30)
            
            if r.status_code == 200: 
                return r.json()['choices'][0]['message']['content']
            elif r.status_code == 429:
                # Rate limited. Rotate key and sleep exponentially to cool down APIs.
                sleep_time = 2 + (attempt * 1.5)
                time.sleep(sleep_time)
                continue
            else: 
                print(f"[{attempt}] API Error {r.status_code} on {model}: {r.text[:50]}")
                time.sleep(2)
                continue
        except Exception as e: 
            time.sleep(2)
            continue
            
    print(f"Exhausted {max_retries} retries for {model}")
    return None

def generate_distractors(sys, user, strategy):
    # Throttle slightly to prevent initial burst from triggering 429s
    time.sleep(1.0)
    if strategy == 'zs': return call_api('CEREBRAS', sys, user, 'llama3.1-8b')
    if strategy == 'cot_en': return call_api('GROQ', sys, user, 'llama-3.3-70b-versatile')
    return call_api('GROQ', sys, user, 'llama-3.1-8b-instant')

def run_audit(sys, user):
    return call_api('GROQ', sys, user, 'llama-3.1-8b-instant')

In [ ]:
# Cell 4: Data Loading
PILOT_N = 50
LANGUAGES = ['English', 'Yoruba', 'Arabic']
dfs = {}
for lang in LANGUAGES:
    for p in Path('/kaggle/input').rglob(f'{lang}_cleaned.csv'):
        df = pd.read_csv(p).rename(columns={'Source_Text_Yo':'source_proverb', 'Source_Text_Mid':'source_proverb', 'Proverb':'source_proverb', 'source_text':'source_proverb',
                                            'Target_Text_En':'proverb_en', 'Translation':'proverb_en', 'english_translation':'proverb_en', 'English_Translation':'proverb_en',
                                            'Cultural_Context':'correct_meaning', 'Correct_Meaning':'correct_meaning'})
        if lang == 'English':
            if 'proverb_en' not in df.columns: df['proverb_en'] = df['source_proverb']
            if 'correct_meaning' not in df.columns: df['correct_meaning'] = df['proverb_en']
        clean_df = df.dropna(subset=['source_proverb', 'proverb_en', 'correct_meaning'])
        if not clean_df.empty:
            dfs[lang] = clean_df.sample(min(PILOT_N, len(clean_df)), random_state=42)
            print(f"Loaded {lang}: {len(dfs[lang])} samples")
            break

In [ ]:
# Cell 5: Master Multi-Model Grid Execution Loop
TASKS, STRATEGIES = ['a', 'b'], ['zs', 'cot_en', 'cot_xl']
all_srs_results, per_lang_master = [], {l: [] for l in LANGUAGES}

for task_id in TASKS:
    for strat_id in STRATEGIES:
        print(f"\n>>> {task_id.upper()}-{strat_id.upper()}")
        task_rows = []
        for lang, df in dfs.items():
            print(f"[{lang}] Generating...", end=" ")
            sys_gen = get_sys_prompt(task_id, strat_id, lang)
            for i, (_, row) in enumerate(df.iterrows()):
                if i % 5 == 0: print(f"{i}...", end=" ", flush=True)
                correct = str(row['correct_meaning']) if task_id == 'b' else str(row['proverb_en'])
                user_gen = f"Proverb: {row['source_proverb']}\nCorrect Answer: {correct}\nReturn JSON."
                raw = generate_distractors(sys_gen, user_gen, strat_id)
                reasoning, dists = parse_response(raw)
                choices, ans = assemble_mcq(correct, dists, i)
                entry = {
                    'task': task_id, 
                    'strategy': strat_id, 
                    'lang': lang, 
                    'source_proverb': row['source_proverb'], 
                    'correct_text': correct,
                    'correct_answer': ans, 
                    'reasoning': reasoning, 
                    **choices
                }
                task_rows.append(entry)
                per_lang_master[lang].append(entry)
            print("Done")
        
        df_gen = pd.DataFrame(task_rows)
        gen_path = OUT_DIR / f"pilot_v5_gen_{task_id}_{strat_id}.csv"
        df_gen.to_csv(gen_path, index=False)
        
        print(f"--- AUDIT {task_id.upper()}-{strat_id.upper()} ---")
        target_type = "translation" if task_id == 'a' else "interpretation"
        audit_sys = f"Reply ONLY with A, B, C, or D. Guess the human {target_type}."
        aud_rows = []
        for _, r in df_gen.iterrows():
            user_aud = f"Which is correct?\nA. {r['Choice_A']}\nB. {r['Choice_B']}\nC. {r['Choice_C']}\nD. {r['Choice_D']}"
            pred = (run_audit(audit_sys, user_aud) or "").strip().upper()[:1]
            hit = int(pred == r['correct_answer'])
            aud_rows.append({'lang': r['lang'], 'hit': hit})
        
        df_aud = pd.DataFrame(aud_rows)
        df_aud.to_csv(OUT_DIR / f"pilot_v5_audit_{task_id}_{strat_id}.csv", index=False)
        for lang in LANGUAGES:
            sub = df_aud[df_aud['lang']==lang]
            if not sub.empty:
                srs = round(sub['hit'].mean() * 100, 1)
                print(f"{lang:<10} SRS: {srs}%")
                all_srs_results.append({'Task': task_id, 'Strat': strat_id, 'Lang': lang, 'SRS': srs})

for lang in LANGUAGES:
    if per_lang_master[lang]: 
        pd.DataFrame(per_lang_master[lang]).to_csv(OUT_DIR / f"pilot_v5_FULL_GEN_{lang}.csv", index=False)
        print(f"Saved: pilot_v5_FULL_GEN_{lang}.csv")

print("\n\nGRID SUMMARY:")
final_df = pd.DataFrame(all_srs_results)
if not final_df.empty:
    pivot = final_df.pivot(index=['Task','Strat'], columns='Lang', values='SRS')
    print(pivot)
    pivot.to_csv(OUT_DIR / "pilot_v5_srs_summary.csv")

In [ ]:
# Cell 6: Final Verification & Zipping
import zipfile
print("\nFILES IN WORKING DIRECTORY:")
for f in OUT_DIR.glob("pilot_v5_*.csv"): print(f"- {f.name}")

zip_name = "proverbgap_pilot_v5_results.zip"
with zipfile.ZipFile(OUT_DIR / zip_name, 'w') as zipf:
    for file in OUT_DIR.glob("pilot_v5_*.csv"): zipf.write(file, file.name)
print(f"\nSUCCESS! Results zipped in {zip_name}")